[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day12-hf-transformers-generate-deep-dive.ipynb)

# Day 12 — HF transformers `generate()` Deep Dive

**Time:** ~30-60 min · **Budget:** CPU or single T4 · Run cells top-to-bottom, no edits.

In [ ]:
%pip install -q transformers torch
# Expected: install completes silently.

## 0. Setup: model + tokenizer

Load GPT-2 (124M). ~548 MB download, runs on CPU or a T4.

In [ ]:
import time, inspect, torch
from transformers import (GPT2LMHeadModel, GPT2TokenizerFast,
    LogitsProcessor, LogitsProcessorList, StoppingCriteria, StoppingCriteriaList,
    RepetitionPenaltyLogitsProcessor)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
tok = GPT2TokenizerFast.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device).eval()
print("Model:", model.config.n_layer, "layers,", model.config.n_head, "heads, head_dim", model.config.n_embd // model.config.n_head)
# Expected: Device: cpu (or cuda) / Model: 12 layers, 12 heads, head_dim 64

## 1. Read the source: where the six stages live

`generate()` dispatches to `sample()` / `greedy_search()` in `transformers/generation/utils.py`.

In [ ]:
from transformers import GenerationMixin
path = inspect.getsourcefile(GenerationMixin)
print(path)
src = inspect.getsource(GenerationMixin.sample)
for i, line in enumerate(src.splitlines()):
    if any(k in line for k in ("logits_processor", "multinomial", "stopping_criteria", "cache_position")):
        print(f"{i:4d}  {line.strip()}")
# Expected: the utils.py path, then lines showing (in order)
# logits_processor call -> warping -> multinomial -> stopping_criteria -> cache_position update

## 2. Verify the repetition-penalty arithmetic (§3.3)

Logits `[5.0, 4.5, 3.0]` with token 0 already generated, penalty 1.2: expect `[4.1667, 4.5, 3.0]` — the flip.

In [ ]:
proc = RepetitionPenaltyLogitsProcessor(penalty=1.2)
logits = torch.tensor([[5.0, 4.5, 3.0]])
input_ids = torch.tensor([[0]])  # token 0 already generated
out = proc(input_ids, logits)
print(out)
print("Ranking flips:", out[0,1] > out[0,0], "(a=%.4f > the=%.4f)" % (out[0,1], out[0,0]))
# Expected: tensor([[4.1667, 4.5000, 3.0000]]) and Ranking flips: True

## 3. Custom `LogitsProcessor`: ban a token

Set the chosen token's logit to -inf. Generate with and without the ban; the token must never appear with the ban on.

In [ ]:
class BanTokenLogitsProcessor(LogitsProcessor):
    def __init__(self, banned_id):
        self.banned_id = banned_id
    def __call__(self, input_ids, scores):
        scores[:, self.banned_id] = -float("inf")
        return scores

prompt = "The cat sat on the"
banned = tok.encode(" the", add_special_tokens=False)[0]
print("Banned token id:", banned, "->", repr(tok.decode([banned])))
inp = tok(prompt, return_tensors="pt").input_ids.to(device)

with torch.no_grad():
    plain = model.generate(inp, max_new_tokens=30, do_sample=True, temperature=0.8,
                           pad_token_id=tok.eos_token_id)
    banned_out = model.generate(inp, max_new_tokens=30, do_sample=True, temperature=0.8,
                                logits_processor=LogitsProcessorList([BanTokenLogitsProcessor(banned)]),
                                pad_token_id=tok.eos_token_id)
print("Without ban:", repr(tok.decode(plain[0])))
print("With ban   :", repr(tok.decode(banned_out[0])))
print("Banned id present without:", (plain[0] == banned).any().item(),
      "| with:", (banned_out[0] == banned).any().item())
# Expected: banned id appears in the plain output (usually), and 0 times with the ban.
# With ban: exactly False

## 4. Custom `StoppingCriteria`: a wall-clock budget

`MaxTimeCriteria` halts generation after N seconds. Request 500 tokens with a 3 s cap.

In [ ]:
class MaxTimeCriteria(StoppingCriteria):
    def __init__(self, max_seconds):
        self.max_seconds = max_seconds
        self.start = time.time()
    def __call__(self, input_ids, scores, **kwargs):
        return (time.time() - self.start) > self.max_seconds

t0 = time.time()
with torch.no_grad():
    out = model.generate(inp, max_new_tokens=500,
                         stopping_criteria=StoppingCriteriaList([MaxTimeCriteria(3.0)]),
                         pad_token_id=tok.eos_token_id)
dt = time.time() - t0
new_tokens = out.shape[1] - inp.shape[1]
print(f"Generated {new_tokens} tokens in {dt:.2f}s (asked for 500, cap 3.0s)")
# Expected: dt ~= 3.0-3.5 s, new_tokens << 500

## 5. Inspect `past_key_values`: the cache, productized

`return_dict_in_generate=True` + `output_scores=True` exposes the loop's internals.

In [ ]:
with torch.no_grad():
    out = model.generate(inp, max_new_tokens=10, return_dict_in_generate=True,
                         output_scores=True, pad_token_id=tok.eos_token_id)
cache = out.past_key_values
print("Cache class:", type(cache).__name__)
k0 = cache.key_cache[0]
print("Layer-0 K shape:", tuple(k0.shape))  # (batch, heads, seq, head_dim)
T = k0.shape[2]
L, H, D, BYTES = model.config.n_layer, model.config.n_head, model.config.n_embd // model.config.n_head, k0.element_size()
kv_bytes = 2 * L * T * H * D * BYTES
print(f"KV bytes at T={T}: {kv_bytes/1e6:.2f} MB  (2*L*T*H*D*bytes)")
print("Num score tensors:", len(out.scores), "| one score shape:", tuple(out.scores[0].shape))
# Expected: Cache class: DynamicCache
# Layer-0 K shape: (1, 12, T, 64) with T = prompt_len + 10
# KV bytes: ~1.1 MB (fp32, T=15: 2*12*15*12*64*4 = 1.11 MB); 10 score tensors of shape (1, 50257)

## 6. Temperature + top-p, computed and verified (§3.4)

Hand-computed: T=0.7 on `[5.0, 4.0, 3.5, 1.0]` -> leader ~73.5%; top-p 0.9 keeps 2 tokens, renormalized to 80.7% / 19.3%.

In [ ]:
from transformers.generation import TemperatureLogitsWarper, TopPLogitsWarper
logits = torch.tensor([[5.0, 4.0, 3.5, 1.0]])
w = TemperatureLogitsWarper(temperature=0.7)
p = torch.softmax(w(None, logits.clone()), dim=-1)
print("T=0.7 probs:", [f"{x:.1%}" for x in p[0].tolist()])
# top-p: keep smallest set with cumulative mass >= 0.9
top_p = TopPLogitsWarper(top_p=0.9)
filtered = torch.softmax(top_p(None, w(None, logits.clone())), dim=-1)
print("After top-p 0.9:", [f"{x:.1%}" for x in filtered[0].tolist()])
# Expected: T=0.7 -> [73.5%, 17.6%, 8.6%, 0.2%]
# top-p 0.9 -> [80.7%, 19.3%, 0.0%, 0.0%]

## 7. Greedy vs sampled throughput

Warping + multinomial add kernels per step. Time 50 tokens each way.

In [ ]:
def tok_per_s(**kw):
    t0 = time.time()
    with torch.no_grad():
        o = model.generate(inp, max_new_tokens=50, pad_token_id=tok.eos_token_id, **kw)
    dt = time.time() - t0
    return (o.shape[1] - inp.shape[1]) / dt

g = tok_per_s(do_sample=False)
s = tok_per_s(do_sample=True, top_p=0.9, temperature=0.8)
print(f"Greedy: {g:.1f} tok/s | Sampled (top-p 0.9): {s:.1f} tok/s")
# Expected: greedy slightly faster; record both in your §4 measure table.

## Done

Checklist: six stages located in source ✓ · penalty flip verified ✓ · ban works ✓ · time cap works ✓ · cache shapes + bytes ✓ · warping math ✓ · tok/s recorded ✓

**Tomorrow (Day 13):** end-to-end inference anatomy — one prompt traced through tokenization, prefill, decode, and detokenization.